# BÀI TẬP LỚN XỬ LÝ NGÔN NGỮ TỰ NHIÊN (NLP)
## Đề tài: Hệ thống Tìm kiếm văn bản & Phân loại tin tức tiếng Việt

**Quy trình triển khai (Pipeline 6 bước đầy đủ):**
1. **Bước 1:** Thu thập dữ liệu tin tức tiếng Việt (VnExpress) bằng `requests` + `BeautifulSoup` & Khảo sát dữ liệu (EDA: số lượng tài liệu, **số câu**, số token, độ dài, dữ liệu thiếu/trùng lặp).
2. **Bước 2:** Tiền xử lý văn bản tiếng Việt (Chuẩn hóa Unicode `NFC`, **ẩn thông tin cá nhân** như email/số điện thoại/URL bằng Regex, tách từ ghép `underthesea`, lọc Stopwords) & Thống kê Vocabulary.
3. **Bước 3:** Chia tập dữ liệu `Train / Validation / Test` có kiểm soát (`stratify`, cố định `random_state=42`, dùng `Pipeline` để tránh Data Leakage).
4. **Bước 4 & 5:** Xây dựng mô hình & Thiết kế thực nghiệm đối chứng (Ablation Study):
   - *Nhánh A (Phân loại tin tức):* E1–E5 (Âm tiết vs Từ ghép, Count vs TF-IDF, Unigram vs Bigram, Multinomial Naive Bayes vs Logistic Regression).
   - *Nhánh B (Mini Search Engine):* S1–S3 (Count vs TF-IDF L2, Unigram vs Bigram + Cosine Similarity).
5. **Bước 6:** Đánh giá chi tiết (Macro-F1, Precision, Recall, Confusion Matrix, Precision@5, MRR@5), Phân tích lỗi (Error Analysis: 10 ví dụ phân loại sai + các ca khó OOV/đồng nghĩa/đa nghĩa) & Demo trực tiếp.

> **Tính tái lập (Reproducibility):** Toàn bộ phiên bản môi trường, dữ liệu trung gian (`data/`), bảng số liệu và biểu đồ (`results/`) được lưu tự động để phục vụ báo cáo PDF và kiểm chứng lại.


In [ ]:
# Cài đặt thư viện cần thiết (chạy trên Google Colab)
!pip install -q requests beautifulsoup4 lxml underthesea scikit-learn pandas numpy matplotlib seaborn

import os
import re
import sys
import time
import json
import unicodedata
import warnings
from collections import Counter
from datetime import datetime
from importlib.metadata import version as pkg_version

import requests
import bs4
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import underthesea

from underthesea import word_tokenize, sent_tokenize
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Tạo thư mục lưu dữ liệu và kết quả
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Ghi nhận đầy đủ phiên bản Python & các thư viện sử dụng (đảm bảo tính tái lập)
def safe_pkg_version(name, fallback="unknown"):
    try:
        return pkg_version(name)
    except Exception:
        return fallback

env_info = {
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit-learn": sklearn.__version__,
    "underthesea": safe_pkg_version("underthesea"),
    "beautifulsoup4": bs4.__version__,
    "requests": requests.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "random_state": RANDOM_STATE,
    "recorded_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open("results/environment_versions.json", "w", encoding="utf-8") as f:
    json.dump(env_info, f, ensure_ascii=False, indent=2)

print("=== THÔNG TIN MÔI TRƯỜNG & PHIÊN BẢN THƯ VIỆN ===")
for k, v in env_info.items():
    print(f"- {k:16s}: {v}")


---
## BƯỚC 1: Thu thập dữ liệu (VnExpress) & Khảo sát dữ liệu (EDA)

- **Nguồn dữ liệu:** 5 chuyên mục công khai trên báo điện tử VnExpress (`Công nghệ`, `Thể thao`, `Kinh tế`, `Giáo dục`, `Sức khỏe`), ngôn ngữ tiếng Việt.
- **Đơn vị mẫu:** Mỗi mẫu là 01 bài báo gồm tiêu đề (`title`), mô tả ngắn (`description`) và nội dung chính (`content`).
- **Quy tắc lọc & đạo đức dữ liệu:** Chỉ thu thập bài báo có tiêu đề và nội dung $\ge 30$ từ; có độ trễ giữa các request (`time.sleep`) để tôn trọng máy chủ; chỉ dùng cho mục đích học thuật và ẩn mọi thông tin liên hệ cá nhân (email, số điện thoại) ở bước tiền xử lý.
- **Cấu hình quy mô:**
  - Chạy thử nhanh: `MAX_PAGES_PER_CAT = 3`, `MAX_ARTICLES_PER_CAT = 60` (~300 bài).
  - Chạy làm báo cáo chính thức: `MAX_PAGES_PER_CAT = 10`–`15`, `MAX_ARTICLES_PER_CAT = 300`–`400` (~1.500–2.000 bài).


In [ ]:
CATEGORIES = {
    "Cong_nghe": "https://vnexpress.net/so-hoa",
    "The_thao":  "https://vnexpress.net/the-thao",
    "Kinh_te":   "https://vnexpress.net/kinh-doanh",
    "Giao_duc":  "https://vnexpress.net/giao-duc",
    "Suc_khoe":  "https://vnexpress.net/suc-khoe",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

MAX_PAGES_PER_CAT = 10      # Tăng lên 15-20 nếu muốn thu thập 2.000 - 3.000 bài
MAX_ARTICLES_PER_CAT = 300
RAW_DATA_PATH = "data/vnexpress_raw.csv"

def get_article_links(category_url, max_pages=5):
    links = []
    seen = set()
    for page in range(1, max_pages + 1):
        url = category_url if page == 1 else f"{category_url}-p{page}"
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            if resp.status_code != 200:
                continue
            soup = BeautifulSoup(resp.text, "lxml")
            for tag in soup.select("h3.title-news a, h2.title-news a"):
                href = tag.get("href", "").strip()
                if href.startswith("https://vnexpress.net/") and href.endswith(".html") and href not in seen:
                    seen.add(href)
                    links.append(href)
            time.sleep(0.25)
        except Exception as e:
            print(f"[Warn] Lỗi trang {url}: {e}")
    return links

def parse_article(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, "lxml")

        title_tag = soup.find("h1", class_="title-detail")
        desc_tag = soup.find("p", class_="description")
        body_paragraphs = soup.select("article.fck_detail p.Normal")

        title = title_tag.get_text(" ", strip=True) if title_tag else ""
        description = desc_tag.get_text(" ", strip=True) if desc_tag else ""
        content = " ".join(p.get_text(" ", strip=True) for p in body_paragraphs)

        if not title or len(content.split()) < 30:
            return None

        return {
            "url": url,
            "title": title,
            "description": description,
            "content": content,
            "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
    except Exception:
        return None

# Nếu đã có file raw thì nạp lại để tiết kiệm thời gian; ngược lại tiến hành thu thập
if os.path.exists(RAW_DATA_PATH):
    print(f"Đã tìm thấy dữ liệu thô tại {RAW_DATA_PATH}. Đang tải lại...")
    df_raw = pd.read_csv(RAW_DATA_PATH)
else:
    records = []
    for cat_name, cat_url in CATEGORIES.items():
        print(f"Đang thu thập chuyên mục: {cat_name} ...")
        links = get_article_links(cat_url, max_pages=MAX_PAGES_PER_CAT)
        print(f"  -> Tìm thấy {len(links)} liên kết. Đang tải nội dung...")
        count_ok = 0
        for link in links:
            if count_ok >= MAX_ARTICLES_PER_CAT:
                break
            art = parse_article(link)
            if art:
                art["category"] = cat_name
                records.append(art)
                count_ok += 1
            time.sleep(0.15)
        print(f"  -> Hoàn thành {cat_name}: {count_ok} bài hợp lệ.")

    df_raw = pd.DataFrame(records)
    df_raw.to_csv(RAW_DATA_PATH, index=False, encoding="utf-8-sig")
    print(f"Đã lưu tổng cộng {len(df_raw)} bài báo vào {RAW_DATA_PATH}")

print("Kích thước bảng dữ liệu thô:", df_raw.shape)
df_raw.head(3)


In [ ]:
# === KHẢO SÁT DỮ LIỆU (EDA): TÀI LIỆU / CÂU / TOKEN / THIẾU / TRÙNG LẶP ===
print("1. Kiểm tra giá trị thiếu (Missing values) trên dữ liệu thô:")
print(df_raw.isnull().sum())

df = df_raw.copy()
df["title"] = df["title"].fillna("")
df["description"] = df["description"].fillna("")
df["content"] = df["content"].fillna("")

n_before = len(df)
df = df.drop_duplicates(subset=["url"]).drop_duplicates(subset=["title"]).reset_index(drop=True)
n_after = len(df)
print(f"\n2. Loại bỏ trùng lặp: {n_before} -> {n_after} bài (đã xóa {n_before - n_after} bản ghi trùng).")

# Ghép tiêu đề + mô tả + nội dung thành văn bản đầy đủ
df["full_text"] = (df["title"] + ". " + df["description"] + " " + df["content"]).str.strip()

# Đếm số câu bằng underthesea.sent_tokenize, số âm tiết thô và số ký tự
df["sentence_count"] = df["full_text"].apply(lambda x: len(sent_tokenize(str(x))))
df["word_count_raw"] = df["full_text"].apply(lambda x: len(str(x).split()))
df["char_count_raw"] = df["full_text"].apply(len)

# Bảng thống kê tổng quan toàn bộ Corpus
corpus_overview = pd.DataFrame([{
    "Tổng số tài liệu": len(df),
    "Số chuyên mục (nhãn)": df["category"].nunique(),
    "Tổng số câu": int(df["sentence_count"].sum()),
    "TB câu / bài": round(df["sentence_count"].mean(), 1),
    "Tổng số âm tiết thô": int(df["word_count_raw"].sum()),
    "TB âm tiết / bài": round(df["word_count_raw"].mean(), 1),
    "Trung vị âm tiết / bài": round(df["word_count_raw"].median(), 1),
    "Ngôn ngữ": "Tiếng Việt (vi)"
}])
corpus_overview.to_csv("results/eda_corpus_overview.csv", index=False, encoding="utf-8-sig")
print("\n3. Thống kê tổng quan toàn bộ Corpus:")
display(corpus_overview)

# Bảng thống kê chi tiết theo chuyên mục
eda_summary = df.groupby("category").agg(
    num_docs=("url", "count"),
    total_sentences=("sentence_count", "sum"),
    avg_sentences=("sentence_count", "mean"),
    avg_words=("word_count_raw", "mean"),
    median_words=("word_count_raw", "median"),
    min_words=("word_count_raw", "min"),
    max_words=("word_count_raw", "max"),
).round(1)
eda_summary["ratio_%"] = (eda_summary["num_docs"] / len(df) * 100).round(2)
eda_summary.to_csv("results/eda_summary.csv", encoding="utf-8-sig")
print("\n4. Bảng thống kê dữ liệu theo từng chuyên mục:")
display(eda_summary)

# Vẽ biểu đồ phân bố lớp, phân bố số câu và phân bố độ dài từ
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.countplot(data=df, x="category", order=eda_summary.index, ax=axes[0], palette="viridis")
axes[0].set_title("Phân bố số bài báo theo chuyên mục")
axes[0].set_xlabel("Chuyên mục")
axes[0].set_ylabel("Số bài báo")
axes[0].tick_params(axis="x", rotation=20)

sns.histplot(df["sentence_count"], bins=35, kde=True, ax=axes[1], color="#2f855a")
axes[1].set_title("Phân bố số câu mỗi bài báo")
axes[1].set_xlabel("Số câu")
axes[1].set_ylabel("Tần suất")

sns.histplot(df["word_count_raw"], bins=40, kde=True, ax=axes[2], color="#2b6cb0")
axes[2].set_title("Phân bố độ dài bài báo (âm tiết thô)")
axes[2].set_xlabel("Số âm tiết")
axes[2].set_ylabel("Tần suất")

plt.tight_layout()
plt.savefig("results/eda_overview.png", dpi=150)
plt.show()


---
## BƯỚC 2: Tiền xử lý văn bản tiếng Việt & Ẩn thông tin cá nhân (Module 3)

Các bước thực hiện có lý giải kỹ thuật:
1. **Chuẩn hóa Unicode (`NFC`):** Đưa toàn bộ dấu tiếng Việt về dạng dựng sẵn nhất quán.
2. **Ẩn thông tin cá nhân (PII Masking) & làm sạch bằng Regex:**
   - Phát hiện và xóa/ẩn **URL**, **địa chỉ Email**, **số điện thoại Việt Nam** (dạng `0xx...`, `+84...`, có thể kèm dấu gạch ngang/chấm/khoảng trắng).
   - **Giữ nguyên dấu tiếng Việt** (không xóa dấu vì làm mất nghĩa từ).
3. **Tạo 2 phiên bản tokenization để đối chứng:**
   - `text_syllable`: Tách theo khoảng trắng (âm tiết).
   - `text_segmented`: Tách từ ghép tiếng Việt bằng `underthesea` (ví dụ `trí_tuệ_nhân_tạo`, `chứng_khoán`).
4. **Stopwords tiếng Việt:** Loại bỏ các hư từ/từ nối phổ biến ít mang tính phân biệt chuyên mục.


In [ ]:
VIETNAMESE_STOPWORDS = {
    "và", "của", "là", "có", "được", "trong", "cho", "với", "các", "những",
    "một", "này", "đó", "khi", "để", "từ", "tại", "về", "như", "theo",
    "đã", "đang", "sẽ", "cũng", "nhưng", "vì", "nên", "nếu", "thì", "mà",
    "hay", "hoặc", "do", "bởi", "vào", "ra", "lên", "xuống", "qua", "lại",
    "trên", "dưới", "giữa", "sau", "trước", "cùng", "còn", "chỉ", "rất",
    "hơn", "nhất", "nhiều", "ít", "mới", "rồi", "đều", "vẫn", "chưa", "phải",
    "ông", "bà", "anh", "chị", "tôi", "chúng_tôi", "họ", "mình", "người",
    "nói", "cho_biết", "theo_đó", "ngoài_ra", "tuy_nhiên", "bên_cạnh"
}

# Biểu thức chính quy ẩn thông tin cá nhân (PII) và nhiễu web
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", flags=re.IGNORECASE)
PHONE_PATTERN = re.compile(
    r"(?:\+84|84|0)(?:[\s\.\-]?\d){8,10}\b"
)

def mask_pii_and_web(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)
    text = PHONE_PATTERN.sub(" ", text)
    return text

# Thống kê số lượng PII (email, số điện thoại, URL) phát hiện & đã được ẩn trong corpus
pii_stats = {
    "URL_masked": int(df["full_text"].apply(lambda x: len(URL_PATTERN.findall(str(x)))).sum()),
    "Email_masked": int(df["full_text"].apply(lambda x: len(EMAIL_PATTERN.findall(str(x)))).sum()),
    "Phone_masked": int(df["full_text"].apply(lambda x: len(PHONE_PATTERN.findall(str(x)))).sum()),
}
print("Thống kê thông tin web / cá nhân (PII) được phát hiện và ẩn:", pii_stats)

def normalize_text(text):
    text = mask_pii_and_web(text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_stopwords(text, stopwords_set=VIETNAMESE_STOPWORDS):
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords_set and len(t) > 1 and not t.isdigit()]
    return " ".join(tokens)

def segment_vi(text):
    text_clean = mask_pii_and_web(text)
    seg = word_tokenize(text_clean, format="text").lower()
    seg = re.sub(r"[^\w\s]", " ", seg, flags=re.UNICODE)
    seg = re.sub(r"\s+", " ", seg).strip()
    return remove_stopwords(seg)

print("Đang chuẩn hóa và tách từ tiếng Việt (có thể mất 1-3 phút tùy số lượng bài)...")
t0 = time.time()

df["text_norm"] = df["full_text"].apply(normalize_text)
df["text_syllable"] = df["text_norm"].apply(remove_stopwords)
df["text_segmented"] = df["full_text"].apply(segment_vi)

print(f"Hoàn thành tiền xử lý sau {time.time() - t0:.1f} giây.")

# Lưu dữ liệu sau tiền xử lý (kết quả trung gian quan trọng)
df.to_csv("data/vnexpress_processed.csv", index=False, encoding="utf-8-sig")

# Thống kê so sánh Vocabulary trước/sau tiền xử lý và giữa âm tiết vs từ ghép
def vocab_stats(series):
    all_tokens = [tok for doc in series for tok in str(doc).split()]
    return len(all_tokens), len(set(all_tokens)), Counter(all_tokens).most_common(10)

tok_raw, voc_raw, top_raw = vocab_stats(df["full_text"])
tok_syl, voc_syl, top_syl = vocab_stats(df["text_syllable"])
tok_seg, voc_seg, top_seg = vocab_stats(df["text_segmented"])

vocab_df = pd.DataFrame([
    {"Phiên bản": "1. Văn bản thô (Raw)", "Tổng số token": tok_raw, "Kích thước Vocabulary": voc_raw},
    {"Phiên bản": "2. Chuẩn hóa + Ẩn PII + Tách âm tiết + Stopwords", "Tổng số token": tok_syl, "Kích thước Vocabulary": voc_syl},
    {"Phiên bản": "3. Chuẩn hóa + Ẩn PII + Tách từ ghép (underthesea)", "Tổng số token": tok_seg, "Kích thước Vocabulary": voc_seg},
])
vocab_df.to_csv("results/eda_vocab_comparison.csv", index=False, encoding="utf-8-sig")
display(vocab_df)

print("\nMinh họa 1 mẫu trước và sau tiền xử lý:")
print("- Gốc:      ", df["title"].iloc[0])
print("- Âm tiết:  ", df["text_syllable"].iloc[0][:150], "...")
print("- Từ ghép:  ", df["text_segmented"].iloc[0][:150], "...")


---
## BƯỚC 3: Chia tập dữ liệu chuẩn mực (Tránh Data Leakage)

- Chia theo tỷ lệ **70% Train / 15% Validation / 15% Test** với `stratify` theo nhãn chuyên mục và cố định `random_state=42`.
- Toàn bộ bước trích xuất đặc trưng (`CountVectorizer`, `TfidfVectorizer`) chỉ được `.fit()` trên tập **Train** thông qua `sklearn.pipeline.Pipeline`.


In [ ]:
# Chia Train (70%) và Temp (30%), sau đó chia Temp thành Validation (15%) và Test (15%)
train_idx, temp_idx = train_test_split(
    df.index, test_size=0.30, stratify=df["category"], random_state=RANDOM_STATE
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=df.loc[temp_idx, "category"], random_state=RANDOM_STATE
)

df_train = df.loc[train_idx].reset_index(drop=True)
df_val   = df.loc[val_idx].reset_index(drop=True)
df_test  = df.loc[test_idx].reset_index(drop=True)

split_stats = pd.DataFrame({
    "Train (70%)": df_train["category"].value_counts(),
    "Validation (15%)": df_val["category"].value_counts(),
    "Test (15%)": df_test["category"].value_counts(),
})
split_stats.loc["TỔNG"] = split_stats.sum()
split_stats.to_csv("results/data_split_stats.csv", encoding="utf-8-sig")
display(split_stats)


---
## BƯỚC 4 & 5 (Nhánh A): Phân loại tin tức & Thực nghiệm đối chứng (E1 – E5)

### 4.1. Tinh chỉnh siêu tham số (Hyperparameter Tuning) CHỈ trên tập Validation
- Với **Multinomial Naive Bayes**: khảo sát hệ số làm mịn Laplace `alpha` $\in \{0.1, 0.5, 1.0, 2.0\}$.
- Với **Logistic Regression**: khảo sát hệ số điều chuẩn `C` $\in \{0.1, 1.0, 5.0, 10.0\}$.


In [ ]:
# 1. Tuning alpha cho MultinomialNB trên Validation
nb_tuning_results = []
for alpha in [0.1, 0.5, 1.0, 2.0]:
    pipe = Pipeline([
        ("vec", CountVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", MultinomialNB(alpha=alpha))
    ])
    pipe.fit(df_train["text_segmented"], df_train["category"])
    pred_val = pipe.predict(df_val["text_segmented"])
    _, _, f1_macro, _ = precision_recall_fscore_support(
        df_val["category"], pred_val, average="macro", zero_division=0
    )
    nb_tuning_results.append({"Model": "MultinomialNB", "Param": f"alpha={alpha}", "Val_Macro_F1": round(f1_macro, 4)})

# 2. Tuning C cho LogisticRegression trên Validation
lr_tuning_results = []
for C in [0.1, 1.0, 5.0, 10.0]:
    pipe = Pipeline([
        ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9)),
        ("clf", LogisticRegression(C=C, max_iter=1000, random_state=RANDOM_STATE))
    ])
    pipe.fit(df_train["text_segmented"], df_train["category"])
    pred_val = pipe.predict(df_val["text_segmented"])
    _, _, f1_macro, _ = precision_recall_fscore_support(
        df_val["category"], pred_val, average="macro", zero_division=0
    )
    lr_tuning_results.append({"Model": "LogisticRegression", "Param": f"C={C}", "Val_Macro_F1": round(f1_macro, 4)})

df_tuning = pd.DataFrame(nb_tuning_results + lr_tuning_results)
df_tuning.to_csv("results/val_hyperparameter_tuning.csv", index=False, encoding="utf-8-sig")
display(df_tuning)

best_alpha = float(max(nb_tuning_results, key=lambda x: x["Val_Macro_F1"])["Param"].split("=")[1])
best_C = float(max(lr_tuning_results, key=lambda x: x["Val_Macro_F1"])["Param"].split("=")[1])
print(f"Tham số tối ưu chọn từ Validation: best_alpha = {best_alpha}, best_C = {best_C}")


### 4.2. Chạy 5 thí nghiệm đối chứng (E1 – E5) trên CÙNG tập Test
Mỗi thí nghiệm thay đổi đúng 1 thành phần để đo tác động rõ ràng:
- **E1 (Baseline):** Tách âm tiết + Count `(1,1)` + MultinomialNB (`alpha=1.0`)
- **E2:** Tách từ ghép (`underthesea`) + Count `(1,1)` + MultinomialNB (`alpha=1.0`)
- **E3:** Tách từ ghép + Count `(1,2)` + MultinomialNB (`best_alpha`)
- **E4:** Tách từ ghép + TF-IDF `(1,1)` + LogisticRegression (`best_C`)
- **E5 (Cải tiến chính):** Tách từ ghép + TF-IDF `(1,2)`, `min_df=2`, `max_df=0.9` + LogisticRegression (`best_C`)


In [ ]:
experiments = [
    {
        "id": "E1",
        "desc": "Baseline: Âm tiết + Count(1,1) + NB",
        "col": "text_syllable",
        "pipe": Pipeline([
            ("vec", CountVectorizer(ngram_range=(1, 1), min_df=2)),
            ("clf", MultinomialNB(alpha=1.0))
        ])
    },
    {
        "id": "E2",
        "desc": "Từ ghép + Count(1,1) + NB",
        "col": "text_segmented",
        "pipe": Pipeline([
            ("vec", CountVectorizer(ngram_range=(1, 1), min_df=2)),
            ("clf", MultinomialNB(alpha=1.0))
        ])
    },
    {
        "id": "E3",
        "desc": "Từ ghép + Count(1,2) + NB (tuned)",
        "col": "text_segmented",
        "pipe": Pipeline([
            ("vec", CountVectorizer(ngram_range=(1, 2), min_df=2)),
            ("clf", MultinomialNB(alpha=best_alpha))
        ])
    },
    {
        "id": "E4",
        "desc": "Từ ghép + TF-IDF(1,1) + LogReg (tuned)",
        "col": "text_segmented",
        "pipe": Pipeline([
            ("vec", TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_df=0.9)),
            ("clf", LogisticRegression(C=best_C, max_iter=1000, random_state=RANDOM_STATE))
        ])
    },
    {
        "id": "E5",
        "desc": "Cải tiến: Từ ghép + TF-IDF(1,2) + LogReg (tuned)",
        "col": "text_segmented",
        "pipe": Pipeline([
            ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9)),
            ("clf", LogisticRegression(C=best_C, max_iter=1000, random_state=RANDOM_STATE))
        ])
    }
]

results_list = []
trained_models = {}
test_predictions = {}

for exp in experiments:
    col = exp["col"]
    pipe = exp["pipe"]
    pipe.fit(df_train[col], df_train["category"])
    y_pred = pipe.predict(df_test[col])

    acc = accuracy_score(df_test["category"], y_pred)
    prec, rec, f1_macro, _ = precision_recall_fscore_support(
        df_test["category"], y_pred, average="macro", zero_division=0
    )
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        df_test["category"], y_pred, average="weighted", zero_division=0
    )
    vocab_size = len(pipe.named_steps["vec"].vocabulary_)

    results_list.append({
        "Mã TN": exp["id"],
        "Cấu hình": exp["desc"],
        "Vocab Size": vocab_size,
        "Accuracy": round(acc, 4),
        "Macro-Precision": round(prec, 4),
        "Macro-Recall": round(rec, 4),
        "Macro-F1": round(f1_macro, 4),
        "Weighted-F1": round(f1_weighted, 4),
    })
    trained_models[exp["id"]] = pipe
    test_predictions[exp["id"]] = y_pred

df_exp_results = pd.DataFrame(results_list)
df_exp_results.to_csv("results/classification_comparison.csv", index=False, encoding="utf-8-sig")
display(df_exp_results)

# Vẽ biểu đồ so sánh Macro-F1 và Accuracy giữa E1-E5
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(df_exp_results))
width = 0.35
ax.bar(x - width/2, df_exp_results["Accuracy"], width, label="Accuracy", color="#4a5568")
ax.bar(x + width/2, df_exp_results["Macro-F1"], width, label="Macro-F1", color="#2b6cb0")
ax.set_xticks(x)
ax.set_xticklabels(df_exp_results["Mã TN"])
ax.set_ylim(0, 1.08)
ax.set_ylabel("Điểm số")
ax.set_title("So sánh hiệu năng phân loại tin tức qua các thí nghiệm E1–E5 (Tập Test)")
ax.legend()
for i, v in enumerate(df_exp_results["Macro-F1"]):
    ax.text(i + width/2, v + 0.015, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig("results/classification_comparison.png", dpi=150)
plt.show()


---
## BƯỚC 4 & 5 (Nhánh B): Hệ thống Tìm kiếm văn bản (Mini Search Engine - Module 4)

So sánh 3 cấu hình Vector Space Model + Cosine Similarity trên cùng kho tài liệu tin tức:
- **S1 (Baseline):** `CountVectorizer(ngram_range=(1,1))` + Cosine Similarity
- **S2:** `TfidfVectorizer(ngram_range=(1,1), norm="l2")` + Cosine Similarity
- **S3 (Cải tiến):** `TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9, norm="l2")` + Cosine Similarity

Đánh giá định lượng bằng **Precision@5** và **MRR@5 (Mean Reciprocal Rank)** trên bộ 15 câu truy vấn chuẩn (3 câu/chuyên mục).


In [ ]:
class MiniSearchEngine:
    def __init__(self, vectorizer, docs_df, text_col="text_segmented"):
        self.vectorizer = vectorizer
        self.docs_df = docs_df.reset_index(drop=True)
        self.text_col = text_col
        self.doc_matrix = self.vectorizer.fit_transform(self.docs_df[text_col])

    def search(self, raw_query, top_k=5):
        processed_q = segment_vi(raw_query)
        q_vec = self.vectorizer.transform([processed_q])
        scores = cosine_similarity(q_vec, self.doc_matrix)[0]
        top_indices = scores.argsort()[::-1][:top_k]

        # Kiểm tra từ OOV trong truy vấn
        vocab = self.vectorizer.vocabulary_
        q_tokens = processed_q.split()
        oov_tokens = [t for t in q_tokens if t not in vocab]

        results = []
        for rank, idx in enumerate(top_indices, start=1):
            results.append({
                "rank": rank,
                "score": round(float(scores[idx]), 4),
                "category": self.docs_df.loc[idx, "category"],
                "title": self.docs_df.loc[idx, "title"],
                "url": self.docs_df.loc[idx, "url"]
            })
        return results, processed_q, oov_tokens

# Khởi tạo 3 phiên bản Search Engine trên toàn bộ kho dữ liệu đã thu thập
search_engines = {
    "S1_Count_1gram": MiniSearchEngine(
        CountVectorizer(ngram_range=(1, 1), min_df=2), df
    ),
    "S2_TFIDF_1gram": MiniSearchEngine(
        TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_df=0.9, norm="l2"), df
    ),
    "S3_TFIDF_12gram": MiniSearchEngine(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, norm="l2"), df
    ),
}

# Bộ 15 câu truy vấn đánh giá định lượng (3 câu mỗi chuyên mục)
EVAL_QUERIES = [
    ("trí tuệ nhân tạo và chip bán dẫn", "Cong_nghe"),
    ("điện thoại thông minh camera mới", "Cong_nghe"),
    ("an ninh mạng và bảo mật dữ liệu", "Cong_nghe"),

    ("bóng đá đội tuyển Việt Nam ghi bàn", "The_thao"),
    ("giải ngoại hạng Anh huấn luyện viên", "The_thao"),
    ("vận động viên huy chương vàng giải đấu", "The_thao"),

    ("lãi suất ngân hàng và tỷ giá đô la", "Kinh_te"),
    ("thị trường chứng khoán cổ phiếu tăng trưởng", "Kinh_te"),
    ("giá vàng và bất động sản xuất khẩu", "Kinh_te"),

    ("tuyển sinh đại học và điểm chuẩn thi tốt nghiệp", "Giao_duc"),
    ("học sinh giáo viên đổi mới chương trình học", "Giao_duc"),
    ("du học học bổng sinh viên nghiên cứu", "Giao_duc"),

    ("phòng chống bệnh tim mạch và huyết áp", "Suc_khoe"),
    ("chế độ dinh dưỡng giảm cân tập thể dục", "Suc_khoe"),
    ("bệnh viện bác sĩ phẫu thuật điều trị", "Suc_khoe"),
]

search_eval_rows = []
top5_sample_rows = []

for name, engine in search_engines.items():
    p_at_5_list = []
    mrr_at_5_list = []
    zero_result_count = 0

    for query, expected_cat in EVAL_QUERIES:
        res, proc_q, oov = engine.search(query, top_k=5)
        hits = [1 if (r["category"] == expected_cat and r["score"] > 0) else 0 for r in res]
        p_at_5 = sum(hits) / 5.0
        mrr = next((1.0 / (i + 1) for i, h in enumerate(hits) if h == 1), 0.0)
        if all(r["score"] == 0 for r in res):
            zero_result_count += 1

        p_at_5_list.append(p_at_5)
        mrr_at_5_list.append(mrr)

        if name == "S3_TFIDF_12gram":
            for r in res[:3]:
                top5_sample_rows.append({
                    "Query": query,
                    "Expected_Cat": expected_cat,
                    "Rank": r["rank"],
                    "Cosine": r["score"],
                    "Pred_Cat": r["category"],
                    "Title": r["title"]
                })

    search_eval_rows.append({
        "Hệ thống": name,
        "Vocab Size": len(engine.vectorizer.vocabulary_),
        "Mean Precision@5": round(float(np.mean(p_at_5_list)), 4),
        "MRR@5": round(float(np.mean(mrr_at_5_list)), 4),
        "Số query trả về 0 điểm": zero_result_count
    })

df_search_eval = pd.DataFrame(search_eval_rows)
df_search_eval.to_csv("results/search_comparison.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(top5_sample_rows).to_csv("results/search_top3_examples.csv", index=False, encoding="utf-8-sig")

print("Bảng so sánh hiệu năng Tìm kiếm văn bản (S1 – S3):")
display(df_search_eval)


---
## BƯỚC 6: Đánh giá chuyên sâu, Phân tích lỗi (Error Analysis) & Demo trực tiếp

### 6.1. Confusion Matrix & Báo cáo chi tiết từng lớp (Baseline E1 vs Cải tiến E5)


In [ ]:
labels_sorted = sorted(df["category"].unique())

print("=== CLASSIFICATION REPORT: BASELINE (E1) ===")
print(classification_report(df_test["category"], test_predictions["E1"], labels=labels_sorted, zero_division=0))

print("\n=== CLASSIFICATION REPORT: MÔ HÌNH CẢI TIẾN (E5) ===")
print(classification_report(df_test["category"], test_predictions["E5"], labels=labels_sorted, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, exp_id, title in [
    (axes[0], "E1", "Confusion Matrix — Baseline (E1)"),
    (axes[1], "E5", "Confusion Matrix — Cải tiến (E5)")
]:
    cm = confusion_matrix(df_test["category"], test_predictions[exp_id], labels=labels_sorted)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_sorted, yticklabels=labels_sorted, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Nhãn dự đoán (Predicted)")
    ax.set_ylabel("Nhãn thật (Actual)")
    ax.tick_params(axis="x", rotation=25)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("results/confusion_matrix_e1_vs_e5.png", dpi=150)
plt.show()


### 6.2. Giải thích mô hình: Top 10 từ khóa đặc trưng nhất theo từng chuyên mục (Trọng số Logistic Regression - E5)


In [ ]:
best_pipe = trained_models["E5"]
feature_names = np.array(best_pipe.named_steps["vec"].get_feature_names_out())
clf_lr = best_pipe.named_steps["clf"]

top_features = {}
for i, cls in enumerate(clf_lr.classes_):
    top_idx = clf_lr.coef_[i].argsort()[::-1][:10]
    top_features[cls] = [f"{feature_names[j]} ({clf_lr.coef_[i][j]:.2f})" for j in top_idx]

df_top_features = pd.DataFrame(top_features)
df_top_features.to_csv("results/top_features_per_class.csv", index=False, encoding="utf-8-sig")
display(df_top_features)


### 6.3. Phân tích lỗi tự động (Xuất 10 ví dụ sai cho Phân loại & Các ca khó cho Tìm kiếm)


In [ ]:
# 1. Phân tích lỗi Phân loại (trên mô hình tốt nhất E5)
probs_e5 = best_pipe.predict_proba(df_test["text_segmented"])
pred_e5 = test_predictions["E5"]

error_rows = []
for i in range(len(df_test)):
    true_y = df_test.loc[i, "category"]
    pred_y = pred_e5[i]
    if true_y != pred_y:
        conf = float(np.max(probs_e5[i]))
        error_rows.append({
            "True_Label": true_y,
            "Pred_Label": pred_y,
            "Confidence": round(conf, 4),
            "Title": df_test.loc[i, "title"],
            "Snippet": df_test.loc[i, "description"][:180]
        })

df_cls_errors = pd.DataFrame(error_rows).sort_values(by="Confidence", ascending=False).head(10)
df_cls_errors.to_csv("results/classification_error_analysis.csv", index=False, encoding="utf-8-sig")
print(f"Tổng số mẫu dự đoán sai trên Test (E5): {len(error_rows)} / {len(df_test)}")
print("Top 10 lỗi có độ tự tin cao nhất (để đưa vào mục Phân tích lỗi trong báo cáo):")
display(df_cls_errors)

# 2. Phân tích lỗi Tìm kiếm (OOV, từ đồng nghĩa, từ đa nghĩa)
STRESS_QUERIES = [
    ("xe hơi điện chạy pin", "Đồng nghĩa (xe hơi vs ô tô)"),
    ("bệnh tiểu đường ăn kiêng", "Đồng nghĩa / thuật ngữ y khoa (đái tháo đường vs tiểu đường)"),
    ("quantum_xyz_999 siêu_máy_tính_ảo", "Từ ngoài từ vựng (OOV)"),
    ("quả táo ra mắt sản phẩm mới", "Từ đa nghĩa / dịch nghĩa đen (Apple vs quả táo)"),
]

search_err_rows = []
best_engine = search_engines["S3_TFIDF_12gram"]
for q, case_type in STRESS_QUERIES:
    res, proc_q, oov = best_engine.search(q, top_k=3)
    top1 = res[0]
    search_err_rows.append({
        "Loại ca khó": case_type,
        "Query gốc": q,
        "Query sau xử lý": proc_q,
        "Từ OOV": ", ".join(oov) if oov else "(Không có)",
        "Top-1 Cosine": top1["score"],
        "Top-1 Chuyên mục": top1["category"],
        "Top-1 Tiêu đề": top1["title"]
    })

df_search_errors = pd.DataFrame(search_err_rows)
df_search_errors.to_csv("results/search_error_analysis.csv", index=False, encoding="utf-8-sig")
print("\nBảng phân tích các ca khó trong Hệ thống Tìm kiếm (OOV / Đồng nghĩa / Đa nghĩa):")
display(df_search_errors)


### 6.4. Demo trực tiếp (Tích hợp Phân loại chuyên mục + Tìm kiếm Top-5 bài báo liên quan)


In [ ]:
def demo_system(user_input, top_k=5):
    print("=" * 80)
    print(f"INPUT: {user_input}")
    print("=" * 80)

    # 1. Dự đoán chuyên mục bằng mô hình E5
    proc_text = segment_vi(user_input)
    probs = best_pipe.predict_proba([proc_text])[0]
    classes = best_pipe.classes_
    ranked_cls = sorted(zip(classes, probs), key=lambda x: x[1], reverse=True)

    print("\n[1] DỰ ĐOÁN CHUYÊN MỤC (TF-IDF 1-2gram + Logistic Regression):")
    for cls, p in ranked_cls[:3]:
        print(f"   - {cls:12s}: {p*100:5.2f}%")

    # 2. Tìm kiếm Top-K bài báo liên quan bằng S3
    res, _, oov = best_engine.search(user_input, top_k=top_k)
    print(f"\n[2] TOP-{top_k} BÀI BÁO LIÊN QUAN NHẤT (TF-IDF + Cosine Similarity):")
    if oov:
        print(f"   (Lưu ý: phát hiện từ ngoài từ vựng OOV: {oov})")
    for r in res:
        print(f"   #{r['rank']} | Cosine={r['score']:.4f} | [{r['category']}] {r['title']}")

# Chạy thử 2 ví dụ demo
demo_system("Apple ra mắt chip AI mới giúp tăng doanh thu quý 4 vượt kỳ vọng")
demo_system("Đội tuyển Việt Nam tập luyện chuẩn bị cho giải đấu bóng đá châu Á")
